In [1]:
import pandas as pd
import re
import os
from google.colab import drive

In [2]:
# 1. Setup
drive.mount('/content/drive')
input_path = '/content/drive/MyDrive/Project/data/filtered_experimental_set.csv'
output_path = '/content/drive/MyDrive/Project/data/perturbed_set.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
def rename_variables(code):
    keywords = {
        'int', 'char', 'float', 'double', 'struct', 'if', 'else', 'while', 'for', 
        'return', 'break', 'continue', 'switch', 'case', 'default', 'sizeof', 
        'static', 'const', 'void', 'unsigned', 'signed', 'long', 'short', 'NULL'
    }

    # This regex matches strings in double quotes OR words
    # Group 1: Strings (to be ignored)
    # Group 2: Identifiers (to be renamed)
    pattern = r'("[^"]*")|(\b[a-zA-Z_][a-zA-Z0-9_]*\b)'

    # 1. First pass: find all valid identifiers NOT in strings
    identifiers = set()
    for match in re.finditer(pattern, code):
        if match.group(2): # If it's a word and not a string
            word = match.group(2)
            if word not in keywords and len(word) > 1:
                identifiers.add(word)

    # 2. Create mapping
    targets = sorted(list(identifiers), key=len, reverse=True)
    mapping = {old: f"var_{i+1}" for i, old in enumerate(targets)}

    # 3. Second pass: Replace only words NOT in strings
    def replace_func(match):
        if match.group(1): # If it's a string, return it untouched
            return match.group(1)
        word = match.group(2)
        return mapping.get(word, word) # Replace if in mapping, else keep

    perturbed_code = re.sub(pattern, replace_func, code)
    return perturbed_code

In [4]:

# 2. Execution
print("--- Starting Perturbation Stage ---")
df = pd.read_csv(input_path)

# Apply renaming to create the perturbed column
df['perturbed_code'] = df['code'].apply(rename_variables)

# Save the final set containing original code, truth, and perturbed code
df.to_csv(output_path, index=False)

print(f"--- Success! Created {len(df)} perturbed samples ---")
print(f"File saved to: {output_path}")

# 3. Quick Preview
print("\n--- Original vs Perturbed Preview ---")
print("ORIGINAL:\n", df.iloc[0]['code'][:100], "...")
print("\nPERTURBED:\n", df.iloc[0]['perturbed_code'][:100], "...")

--- Starting Perturbation Stage ---
--- Success! Created 64 perturbed samples ---
File saved to: /content/drive/MyDrive/Project/data/perturbed_set.csv

--- Original vs Perturbed Preview ---
ORIGINAL:
 ExprResolveLhs(struct xkb_context *ctx, const ExprDef *expr,
               const char **elem_rtrn,  ...

PERTURBED:
 var_1(struct var_5 *var_21, const var_14 *var_19,
               const char **var_11, const char **v ...
